# Production-Ready Warehouse Agent with AgentCore Memory & Evaluation

In Lab 7, we deployed our warehouse operations agent to Amazon Bedrock AgentCore Runtime. But operations managers report two problems:

1. **No memory** — Every session starts from scratch. The agent forgets that a user always cares about Warehouse 1750 and WM-AN02 sensors.
2. **No quality measurement** — We have no way to verify the agent's answers are correct before trusting it for real procurement decisions.

In this lab we solve both:

- **Part A:** Add AgentCore Memory (short-term session continuity + long-term user preferences) so the agent remembers context across conversations.
- **Part B:** Build an evaluation suite using AgentCore Evaluations to measure agent quality with built-in and custom LLM-as-a-judge evaluators.

## Architecture

```
┌───────────────────────────────────────────────────────────────────────┐
│                     Amazon Bedrock AgentCore                           │
│                                                                       │
│  ┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐  │
│  │  Agent Runtime   │    │  AgentCore Memory │    │  Evaluations    │  │
│  │  (from Lab 7)    │◄──►│  - Short-term     │    │  - Built-in     │  │
│  │                  │    │  - Long-term      │    │  - Custom       │  │
│  │  Warehouse Agent │    │  - Semantic       │    │  - On-demand    │  │
│  │  + SAP GenAI Hub │    │  - User Prefs     │    │  - OTEL Traces  │  │
│  └─────────────────┘    └──────────────────┘    └─────────────────┘  │
│           │                                              │            │
│           │              HTTPS (SAP GenAI Hub)           │            │
│           ▼                                              ▼            │
│  ┌─────────────────┐                          ┌─────────────────┐    │
│  │ SAP S/4HANA     │                          │ CloudWatch      │    │
│  │ OData APIs      │                          │ (OTEL Traces)   │    │
│  └─────────────────┘                          └─────────────────┘    │
└───────────────────────────────────────────────────────────────────────┘
```

Note: If you have not set up your ai-core credentials yet, please follow notebook 00-load-sap-ai-core-credentials.

---

## Prerequisites

1. Completed Lab 7
2. SAP AI Core credentials in your `~/.aicore/config.json` file
3. AWS credentials configured with AgentCore Memory and Evaluation permissions
4. SAP S/4HANA Public Cloud API key

In [ ]:
# !pip install .

## 1. Import Dependencies and Initialize Model

In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from util.odata_tool import odata_caller
from strands import Agent, tool
from strands.hooks import (
    AfterInvocationEvent,
    MessageAddedEvent,
    HookProvider,
    HookRegistry,
)

import os
import json
import time
import uuid
from typing import List
from datetime import datetime
from collections import defaultdict

import boto3
from boto3.session import Session
from botocore.exceptions import ClientError
from dotenv import load_dotenv
import getpass

from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import (
    ConversationalMessage,
    MessageRole,
    RetrievalConfig,
    StrategyType,
)
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Load environment variables from .env file
load_dotenv()

# Prompt for SAP API key if not set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")

In [ ]:
# Validate required configuration before proceeding
import os

_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json — run notebook 00 first")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set — check your .env file")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel

# TODO: Choose your model — options: "anthropic--claude-4-sonnet", "amazon--nova-lite", "amazon--nova-pro"
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4-sonnet",
    max_tokens=4096,
)

# Configuration
from boto3.session import Session
from datetime import datetime
import yaml

boto_session = Session()
REGION = boto_session.region_name or "us-east-1"

# TODO: Set a unique actor ID for your user/persona
ACTOR_ID = "warehouse_manager_001"
SESSION_ID = f"warehouse_session_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Load agent configuration from Lab 7 deployment
with open(".bedrock_agentcore.yaml", "r") as f:
    agentcore_config = yaml.safe_load(f)

agent_config = agentcore_config["agents"]["warehouse_ops_agent_sap_genai_hub"]
AGENT_ID = agent_config["bedrock_agentcore"]["agent_id"]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]

# Message role constants
from bedrock_agentcore.memory.constants import MessageRole
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

print(f"Region: {REGION}")
print(f"Actor ID: {ACTOR_ID}")
print(f"Session ID: {SESSION_ID}")
print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"Model: anthropic--claude-4-sonnet via SAP GenAI Hub")

---

# Part A: AgentCore Memory

We add two types of memory to our warehouse agent:

1. **Short-term memory** — Remembers the current conversation session. If the user asks about WM-AN02 and then says "what about the power modules?", the agent maintains context.

2. **Long-term memory** — Persists knowledge across sessions using built-in strategies:
   - **Semantic strategy**: Stores operational facts (e.g., "User manages Warehouse 1750")
   - **User preference strategy**: Captures preferences (e.g., "Show stock as percentage of 500-unit capacity")

We use the `MemorySessionManager` pattern from the AgentCore SDK which handles actor/session scoping automatically.

## 2. Create AgentCore Memory Resource

We create a memory resource with built-in strategies. 

In [ ]:
# Initialize Memory Client
memory_client = MemoryClient(region_name=REGION)

# TODO: Choose a unique memory name for your agent
MEMORY_NAME = "WarehouseAgentMemory"
memory_id = None

# Define strategies using built-in types (no IAM role required)
strategies = [
    {
        StrategyType.SEMANTIC.value: {
            "name": "WarehouseFacts",
            "description": "Stores operational facts about warehouse inventory, orders, and logistics",
            "namespaces": ["warehouse/{actorId}/facts/"],
        }
    },
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "UserPreferences",
            "description": "Stores user preferences like reporting format, products of interest, and alert thresholds",
            "namespaces": ["warehouse/{actorId}/preferences/"],
        }
    },
]

try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Memory for warehouse operations agent - session context and user preferences",
        strategies=strategies,
        event_expiry_days=30,
    )
    memory_id = memory["id"]
    print(f"Created memory resource: {memory_id}")
    print(f"Status: {memory['status']}")

except ClientError as e:
    if "already exists" in str(e):
        memories = memory_client.list_memories()
        memory_id = next(
            (m["id"] for m in memories if m["id"].startswith(MEMORY_NAME)), None
        )
        print(f"Memory already exists: {memory_id}")
    else:
        raise e

print(f"\nMemory ID: {memory_id}")

## 3. Initialize Session Manager

The `MemorySessionManager` provides a cleaner API for session-scoped operations. We create a `MemorySession` for our warehouse manager actor.

In [ ]:
# Initialize the session manager
session_manager = MemorySessionManager(
    memory_id=memory_id, region_name=REGION
)

# Create a memory session for the warehouse manager
warehouse_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID, session_id=SESSION_ID
)

print(f"Session manager initialized for memory: {memory_id}")
print(f"Session created for actor: {ACTOR_ID}")

## 4. Create Memory Hook Provider

The memory hook integrates with the Strands agent lifecycle:
- **On `MessageAddedEvent`**: Retrieves relevant long-term memories when a user message arrives and injects them into the agent's context.
- **On `AfterInvocationEvent`**: Saves the complete interaction (user query + agent response) to memory for future recall.

This follows the pattern from the [AgentCore memory samples](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-features/04-manage-context-of-your-agent/memory).

In [ ]:
class WarehouseMemoryHooks(HookProvider):
    """Memory hooks for warehouse agent using MemorySession."""

    def __init__(self, warehouse_session: MemorySession):
        self.warehouse_session = warehouse_session
        self.retrieval_config = {
            "warehouse/{actorId}/facts/": RetrievalConfig(top_k=5, relevance_score=0.2),
            "warehouse/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
        }

    def retrieve_warehouse_context(self, event: MessageAddedEvent):
        """Retrieve relevant memories when user sends a message."""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]
            memory_context_parts = []

            # Load short-term memory (recent turns from this session)
            try:
                recent_turns = self.warehouse_session.get_last_k_turns(k=5)
                if recent_turns:
                    history_lines = []
                    for turn in recent_turns:
                        for message in turn:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                            history_lines.append(f"{role}: {content}")
                    memory_context_parts.append(
                        "Recent Conversation:\n" + "\n".join(history_lines)
                    )
            except Exception as e:
                print(f"  [Memory] Short-term load failed: {e}")

            # Load long-term memories across namespaces
            try:
                for namespace_template, config in self.retrieval_config.items():
                    resolved_namespace = namespace_template.format(
                        actorId=self.warehouse_session._actor_id
                    )
                    memories = self.warehouse_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k,
                    )
                    filtered = [
                        m for m in memories
                        if m.get("score", 0) >= config.relevance_score
                    ]
                    if filtered:
                        lines = [f"- {m['content']['text']}" for m in filtered[:5]]
                        label = "Facts" if "facts" in namespace_template else "Preferences"
                        memory_context_parts.append(
                            f"Known {label}:\n" + "\n".join(lines)
                        )
            except Exception as e:
                print(f"  [Memory] Long-term load failed: {e}")

            # Inject context into agent's system prompt
            if memory_context_parts:
                context_block = "\n\n".join(memory_context_parts)
                event.agent.system_prompt += (
                    f"\n\n--- MEMORY CONTEXT ---\n{context_block}\n"
                    "Use this context to personalize responses. Do not ask for information "
                    "you already know from memory.\n--- END MEMORY CONTEXT ---"
                )
                print(f"  [Memory] Injected {len(memory_context_parts)} memory sections")

    def save_warehouse_interaction(self, event: AfterInvocationEvent):
        """Save interaction to memory after agent responds."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Find the last user query and assistant response
                user_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        if msg["content"] and msg["content"][0].get("text"):
                            agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not user_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        user_query = msg["content"][0]["text"]
                        break

                if user_query and agent_response:
                    interaction_messages = [
                        ConversationalMessage(user_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT),
                    ]
                    result = self.warehouse_session.add_turns(interaction_messages)
                    print(f"  [Memory] Saved interaction - Event ID: {result['eventId']}")

        except Exception as e:
            print(f"  [Memory] Save failed: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(MessageAddedEvent, self.retrieve_warehouse_context)
        registry.add_callback(AfterInvocationEvent, self.save_warehouse_interaction)


print("WarehouseMemoryHooks defined.")

## 5. Create the Memory-Enhanced Warehouse Agent

We create the warehouse agent with the memory hook attached. Same system prompt and tools as Lab 6, but now with persistent memory.

In [ ]:
# TODO: Customize the system prompt for your warehouse/domain
WAREHOUSE_SYSTEM_PROMPT = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750). 
You have access to real-time SAP warehouse data through OData APIs.

CORE CAPABILITIES:
1. Product Discovery: Find available products and their stock levels
2. Dynamic Querying: Construct intelligent OData queries based on user needs
3. Order Fulfillment: Check if orders can be fulfilled based on current inventory

AVAILABLE PRODUCTS:
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Provide specific, actionable insights with quantitative data
- If you know the user's preferences from memory, apply them without asking
- Do not use emojis

When users ask questions:
1. Determine what data you need
2. Use the odata_caller tool to query SAP S/4HANA APIs
3. Analyze results and provide comprehensive responses

For OData calls, use:
- base_url: "https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata4/sap/api_whse_physstockprod/srvd_a2x/sap/whsephysicalstockproducts/0001"
- auth_type: "api_key"
- auth_env_var: "SAP_S4HANA_PUBLIC_CLOUD_KEY"
"""

# TODO: Adjust capacity for your warehouse (used in evaluation scenarios)
WAREHOUSE_CAPACITY = 500


def create_warehouse_agent(with_memory=True):
    """Create the warehouse agent, optionally with memory hooks."""
    hooks = []
    if with_memory and memory_id:
        hooks.append(WarehouseMemoryHooks(warehouse_session))

    agent = Agent(
        model=model,
        system_prompt=WAREHOUSE_SYSTEM_PROMPT,
        tools=[odata_caller],
        hooks=hooks,
    )
    return agent


# Create the memory-enhanced agent
warehouse_agent = create_warehouse_agent(with_memory=True)
print("Warehouse agent created with memory hooks.")

## 6. Test Memory — First Conversation

Let's have a multi-turn conversation that establishes context. The agent will store these interactions for future recall.

In [ ]:
# Turn 1: Establish user identity and interest
print("User: I'm the operations manager for Warehouse 1750. I primarily track WM-AN02 Control Units.")
print("      Please always show me stock quantities as a percentage of our 500-unit capacity.\n")
response = warehouse_agent(
    "I'm the operations manager for Warehouse 1750. I primarily track WM-AN02 Control Units. "
    "Please always show me stock quantities as a percentage of our 500-unit capacity."
)
print(f"\nAgent: {response.message}")

In [ ]:
# Turn 2: Follow-up that requires session context
print("User: What about the power modules? Are we running low on those too?\n")
response = warehouse_agent("What about the power modules? Are we running low on those too?")
print(f"\nAgent: {response.message}")

In [ ]:
# Turn 3: Establish a reorder threshold preference
print("User: Going forward, alert me whenever any product drops below 20% capacity.")
print("      That's my reorder threshold.\n")
response = warehouse_agent(
    "Going forward, alert me whenever any product drops below 20% capacity. That's my reorder threshold."
)
print(f"\nAgent: {response.message}")

## 7. Test Memory — New Session (Simulating User Returning)

Now we simulate the user coming back. The agent should remember:
- The user is the Warehouse 1750 operations manager
- They primarily care about WM-AN02 Control Units
- They want quantities shown as percentage of 500-unit capacity
- Their reorder threshold is 20%

We wait for long-term memory extraction to process, then create a fresh agent instance.

In [ ]:
# Wait for long-term memory extraction (async process)
print("Waiting 60 seconds for long-term memory extraction...")
time.sleep(60)

# Create a new agent instance (simulates user returning)
warehouse_agent_v2 = create_warehouse_agent(with_memory=True)

print("\nNew agent instance created. Testing memory recall...")
print("\nUser: Any alerts I should know about?\n")
response = warehouse_agent_v2("Any alerts I should know about?")
print(f"\nAgent: {response.message}")

## 8. Inspect Stored Memories

Let's look at what AgentCore Memory has extracted and stored.

In [ ]:
# Short-term memory (recent turns)
print("=== Short-Term Memory (Recent Conversation Turns) ===\n")
try:
    recent_turns = warehouse_session.get_last_k_turns(k=5)
    for i, turn in enumerate(recent_turns, 1):
        print(f"Turn {i}:")
        for message in turn:
            role = message.get("role", "unknown")
            content = message.get("content", {}).get("text", "")[:150]
            full_text = message.get("content", {}).get("text", "")
            print(f"  {role}: {content}{'...' if len(full_text) > 150 else ''}")
        print()
except Exception as e:
    print(f"  Could not retrieve short-term memory: {e}")

# Long-term memory — semantic facts
print("\n=== Long-Term Memory: Semantic Facts ===\n")
try:
    facts = warehouse_session.search_long_term_memories(
        query="warehouse operations inventory products",
        namespace_prefix=f"warehouse/{ACTOR_ID}/facts/",
        top_k=10,
    )
    if facts:
        for i, hit in enumerate(facts, 1):
            print(f"  {i}. {hit['content']['text']}")
    else:
        print("  (No facts extracted yet - extraction is async, may need more time)")
except Exception as e:
    print(f"  Could not retrieve facts: {e}")

# Long-term memory — user preferences
print("\n=== Long-Term Memory: User Preferences ===\n")
try:
    prefs = warehouse_session.search_long_term_memories(
        query="preferences format threshold capacity",
        namespace_prefix=f"warehouse/{ACTOR_ID}/preferences/",
        top_k=10,
    )
    if prefs:
        for i, hit in enumerate(prefs, 1):
            print(f"  {i}. {hit['content']['text']}")
    else:
        print("  (No preferences extracted yet - extraction is async, may need more time)")
except Exception as e:
    print(f"  Could not retrieve preferences: {e}")

---

# Part B: AgentCore Evaluation

Now that we have a memory-enhanced agent deployed to AgentCore Runtime (Lab 7) with OTEL instrumentation, we can measure its quality using **AgentCore Evaluations**.

AgentCore Evaluations works by:
1. The deployed agent emits OpenTelemetry spans to CloudWatch Logs (already configured in Lab 7)
2. We invoke the agent and wait for spans to be ingested (~180 seconds)
3. The `EvaluationClient` reads those spans and scores them using built-in or custom LLM-as-a-judge evaluators

We'll demonstrate:
- **Built-in evaluators** — `Builtin.Correctness`, `Builtin.Helpfulness`, `Builtin.GoalSuccessRate`, `Builtin.ToolSelectionAccuracy`
- **Custom evaluator** — Domain-specific LLM-as-a-judge for SAP warehouse operational quality

```
┌────────────────────────────────────────────────────────────────────────┐
│                        Evaluation Flow                                  │
│                                                                        │
│  ┌──────────┐    invoke    ┌──────────────┐    OTEL    ┌───────────┐  │
│  │ Notebook │────────────► │ Deployed     │──────────► │CloudWatch │  │
│  │ (this)   │             │ Agent (Lab7) │  spans    │ Logs      │  │
│  └──────────┘             └──────────────┘           └─────┬─────┘  │
│       │                                                     │        │
│       │  EvaluationClient.run()                             │        │
│       │◄────────────────────────────────────────────────────┘        │
│       │         reads spans, scores with evaluators                   │
│       ▼                                                              │
│  ┌──────────┐                                                        │
│  │ Scores   │  per-evaluator results (value, label, explanation)     │
│  └──────────┘                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

## 9. Configure Evaluation Infrastructure

Set up the AgentCore Runtime client and helper functions for invoking the deployed agent and waiting for OTEL span ingestion.

In [ ]:
from datetime import timedelta

# AgentCore Runtime client for invoking the deployed agent
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# Derive the CloudWatch log group where OTEL spans land
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

# Ingestion delay — time to wait for OTEL spans to arrive in CloudWatch.
INGESTION_DELAY = 180


def invoke_with_retry(client, agent_arn, session_id, prompt, max_retries=3, wait=30):
    """Invoke agent runtime with retry for cold start 500 errors."""
    for attempt in range(max_retries):
        try:
            response = client.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                runtimeSessionId=session_id,
                payload=json.dumps({"prompt": prompt}).encode("utf-8"),
            )
            return response
        except client.exceptions.RuntimeClientError as e:
            if attempt < max_retries - 1:
                print(f"  Runtime error (attempt {attempt + 1}/{max_retries}), retrying in {wait}s (likely cold start)...")
                time.sleep(wait)
                session_id = f"{session_id}_r{attempt + 1}"
            else:
                raise e
    return None


print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"CloudWatch Log Group: {CW_LOG_GROUP}")
print(f"Ingestion delay: {INGESTION_DELAY}s")

## 10. Create Custom Evaluator

We register a domain-specific LLM-as-a-judge evaluator in the AgentCore control plane. This complements the built-in evaluators with SAP warehouse-specific scoring criteria that generic evaluators cannot assess.

**WarehouseOperationalQuality** (TRACE-level): Evaluates whether the agent's response is operationally useful for a warehouse manager — does it provide actionable inventory insights, use correct product codes, and present data in a way that supports procurement decisions?

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

_SUFFIX = uuid.uuid4().hex[:8]

# TODO: Use the inference profile matching your region (us.* for us-east-1, eu.* for eu-central-1)
JUDGE_MODEL_ID = "us.amazon.nova-pro-v1:0"

# Custom TRACE-level evaluator: Warehouse Operational Quality
print("Creating WarehouseOperationalQuality evaluator (TRACE-level)...")
warehouse_quality_response = agentcore_control.create_evaluator(
    evaluatorName=f"WarehouseOperationalQuality_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "You are a warehouse operations expert evaluating an AI assistant that queries "
                "SAP S/4HANA warehouse APIs for inventory management.\n\n"
                "Conversation context: {context}\n"
                "Agent response: {assistant_turn}\n"
                "Expected behavior: {expected_response}\n\n"
                "Evaluate the OPERATIONAL QUALITY of the response for a warehouse manager. Score based on:\n"
                "1. Does the response contain specific, quantitative inventory data (not vague statements)?\n"
                "2. Are SAP product codes (WM-AN01, WM-AN02, WM-AN03, WM-AN04) used correctly?\n"
                "3. Is the data presented in a way that supports immediate operational decisions "
                "(e.g., reorder recommendations, fulfillment feasibility, capacity utilization)?\n"
                "4. Does the response avoid hallucinating inventory numbers when API data is unavailable?\n\n"
                "Important: If the agent successfully queried the API and returned real data with "
                "correct product codes and actionable insights, score 1.0 even if formatting differs "
                "from the expected response.\n\n"
                "You MUST respond with EXACTLY one of these scores:\n"
                "- 0.0 if the response lacks inventory data, hallucinates numbers, or is not actionable\n"
                "- 0.5 if the response has some useful data but is missing key operational context\n"
                "- 1.0 if the response provides accurate, actionable warehouse intelligence\n\n"
                "Respond with only the numeric score (0.0, 0.5, or 1.0) on the first line, "
                "followed by a one-sentence explanation on the next line."
            ),
            "ratingScale": {
                "numerical": [
                    {"value": 0.0, "label": "not_actionable", "definition": "Response lacks data, hallucinates numbers, or provides no operational value."},
                    {"value": 0.5, "label": "partially_useful", "definition": "Some useful data present but missing key operational context for decisions."},
                    {"value": 1.0, "label": "operationally_excellent", "definition": "Accurate, specific, and actionable warehouse intelligence."},
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": JUDGE_MODEL_ID,
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_EVALUATOR_ID = warehouse_quality_response["evaluatorId"]
print(f"  Created: {CUSTOM_EVALUATOR_ID}")
print(f"\nCustom evaluator registered in AgentCore control plane.")

## 11. Define Evaluation Scenarios

Each scenario contains:
- **prompt**: The user query to send to the deployed agent
- **expected_response**: Description of what a good response looks like (used by Correctness + custom evaluator)
- **expected_trajectory**: Expected tool calls for ToolSelectionAccuracy
- **assertions**: Conditions for GoalSuccessRate
- **max_tool_calls**: Tool-call budget for the trajectory-efficiency grader (section 14b)

The last scenario, `low-stock-items`, is drawn from a **real workshop failure**: the query "What are the items with low stock?" used to trigger many redundant OData calls (repeated `$metadata` discovery, trial-and-error `$filter` guessing) while still producing a correct answer. The built-in `Correctness`/`GoalSuccessRate` evaluators score that path 1.0 — they measure *whether* the answer is right, not *how efficiently* it was reached. We attach a tool-call budget so the efficiency grader in section 14b can catch the wasteful trajectory.

In [ ]:
evaluation_scenarios = [
    {
        "name": "inventory-check-single",
        "prompt": "What is the current stock level for WM-AN02 Control Units?",
        "expected_response": (
            "The response should contain the specific stock quantity for WM-AN02 Control Units "
            "retrieved from the SAP warehouse API, with the product correctly identified as "
            "Control Units. The number should come from actual API data, not be invented."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent correctly identified WM-AN02 as Control Units. "
            "Agent reported a specific numeric stock quantity from the API."
        ),
        # Tool-call budget: one schema-discovery call + one data call is plenty.
        "max_tool_calls": 2,
    },
    {
        "name": "inventory-overview",
        "prompt": "Give me a complete overview of all products currently in the warehouse.",
        "expected_response": (
            "The response should list all warehouse products (WM-AN01 Advanced Sensors, "
            "WM-AN02 Control Units, WM-AN03 Power Modules, WM-AN04 Communication Devices) "
            "with their current stock quantities from the API, presented in a structured format."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via OData. "
            "Agent listed multiple products with stock quantities. "
            "Agent presented results in a structured, readable format."
        ),
        # One unfiltered query returns all products; a second is the most that should be needed.
        "max_tool_calls": 2,
    },
    {
        "name": "fulfillment-feasibility",
        "prompt": "Can we fulfill an order for 200 units of WM-AN02 Control Units?",
        "expected_response": (
            "The response should check current WM-AN02 stock from the API, compare it against "
            "the requested 200 units, and provide a clear yes/no fulfillment recommendation "
            "with the actual available quantity."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried current WM-AN02 stock via OData. "
            "Agent compared available quantity against the 200-unit request. "
            "Agent provided a clear yes/no fulfillment answer with supporting data."
        ),
        "max_tool_calls": 2,
    },
    {
        # Regression scenario drawn from a real workshop failure. This query used to trigger
        # many redundant OData calls (repeated $metadata discovery, trial-and-error $filter
        # guessing). Correctness/GoalSuccessRate scored it 1.0 despite the waste, so we pair it
        # with the tool-call-efficiency grader (section 14b) to catch the inefficient trajectory.
        "name": "low-stock-items",
        "prompt": "What are the items with low stock?",
        "expected_response": (
            "The response should identify which products are low on stock (or state that none "
            "are below the reorder threshold), based on stock quantities from the SAP warehouse "
            "API. The agent should reach the answer efficiently — ideally a single filtered or "
            "sorted OData query — rather than fetching everything and repeatedly rediscovering the schema."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent determined which products are low on stock relative to their capacity. "
            "Agent reached the answer without redundant, repeated OData calls."
        ),
        # Same tight budget: a low-stock check should be one filtered/sorted query.
        "max_tool_calls": 2,
    },
]

print(f"Evaluation scenarios defined: {len(evaluation_scenarios)}")
for s in evaluation_scenarios:
    print(f"  - {s['name']}: {s['prompt']} (budget: {s['max_tool_calls']} tool calls)")

## 12. Invoke Deployed Agent for Each Scenario

We invoke the deployed agent (from Lab 7) for each evaluation scenario. Each invocation gets a unique `runtimeSessionId` so the evaluator can locate its spans independently.

In [ ]:
# Invoke the deployed agent for each scenario and collect session IDs
eval_sessions = []

print("Invoking deployed agent for each evaluation scenario...\n")


def parse_agent_response(response_body: str) -> str:
    """Parse invoke_agent_runtime response into plain text.

    The response is a JSON object: {"role": "assistant", "content": [{"text": "..."}], "metadata": {...}}
    It may arrive as a single blob or as multiple newline-delimited chunks that concatenate into one object.
    """
    text_parts = []

    # Try parsing as a single JSON object (most common)
    try:
        data = json.loads(response_body)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Fall back: response may be multiple concatenated JSON chunks (chunked transfer)
    try:
        combined = "".join(response_body.strip().split("\n"))
        data = json.loads(combined)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Last resort: return raw truncated
    return response_body[:500]


for scenario in evaluation_scenarios:
    session_id = f"eval_{scenario['name']}_{uuid.uuid4().hex}"

    try:
        response = invoke_with_retry(
            agentcore_client, AGENT_ARN, session_id, scenario["prompt"]
        )

        response_body = response["response"].read().decode("utf-8")
        agent_text = parse_agent_response(response_body)

        eval_sessions.append({
            "scenario_name": scenario["name"],
            "session_id": session_id,
            "prompt": scenario["prompt"],
            "response": agent_text[:500],
            "expected_response": scenario["expected_response"],
            "expected_trajectory": scenario["expected_trajectory"],
            "assertions": scenario["assertions"],
            "max_tool_calls": scenario["max_tool_calls"],
        })

        print(f"  [{scenario['name']}] Session: {session_id}")
        print(f"    Response: {agent_text[:100]}...\n")

    except Exception as e:
        print(f"  [{scenario['name']}] ERROR: {e}\n")

print(f"\nCompleted {len(eval_sessions)} agent invocations.")
print(f"Waiting {INGESTION_DELAY}s for CloudWatch span ingestion...")
time.sleep(INGESTION_DELAY)
print("Spans should now be available for evaluation.")

## 13. Evaluate with Built-in Evaluators

We use `EvaluationClient.run()` to score each session with AgentCore's built-in evaluators:

| Evaluator | Level | Needs Ground Truth | What it measures |
|-----------|-------|-------------------|-----------------|
| `Builtin.Correctness` | TRACE | `expected_response` | Factual accuracy of the response |
| `Builtin.Helpfulness` | TRACE | None | How useful/valuable the response is |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` | Whether the agent completed the user's goal |
| `Builtin.ToolSelectionAccuracy` | SESSION | None | Whether the agent chose the right tools |

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

ec = EvaluationClient(region_name=REGION)

# Pre-populate the evaluator level cache (required for the SDK to route correctly)
ec._evaluator_level_cache.update({
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.ToolSelectionAccuracy": "SESSION",
})

BUILTIN_EVALUATOR_IDS = [
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.GoalSuccessRate",
    "Builtin.ToolSelectionAccuracy",
]

builtin_results = {}

print("Running built-in evaluators on each session...\n")

for session in eval_sessions:
    print(f"  Evaluating: {session['scenario_name']} (session: {session['session_id']})")

    try:
        results = ec.run(
            evaluator_ids=BUILTIN_EVALUATOR_IDS,
            agent_id=AGENT_ID,
            session_id=session["session_id"],
            look_back_time=timedelta(hours=1),
            reference_inputs=ReferenceInputs(
                expected_response=session["expected_response"],
                expected_trajectory=session["expected_trajectory"],
                assertions=[session["assertions"]],
            ),
        )
        builtin_results[session["scenario_name"]] = results
        for r in results:
            score = r.get("value", "N/A")
            label = r.get("label", "")
            evaluator = r.get("evaluatorId", "unknown")
            print(f"    {evaluator}: {score} ({label})")
    except Exception as e:
        print(f"    ERROR: {e}")
        builtin_results[session["scenario_name"]] = []

    print()

print("Built-in evaluation complete.")

## 14. Evaluate with Custom Evaluator

Now we run our domain-specific **WarehouseOperationalQuality** evaluator on the same sessions. This TRACE-level evaluator scores whether the agent provides actionable warehouse intelligence — something the generic built-in evaluators cannot assess.

In [ ]:
# Add custom evaluator level to the cache
ec._evaluator_level_cache[CUSTOM_EVALUATOR_ID] = "TRACE"

custom_results = {}

print("Running WarehouseOperationalQuality evaluator on each session...\n")

for session in eval_sessions:
    print(f"  Evaluating: {session['scenario_name']} (session: {session['session_id']})")

    try:
        results = ec.run(
            evaluator_ids=[CUSTOM_EVALUATOR_ID],
            agent_id=AGENT_ID,
            session_id=session["session_id"],
            look_back_time=timedelta(hours=1),
            reference_inputs=ReferenceInputs(
                expected_response=session["expected_response"],
                expected_trajectory=session["expected_trajectory"],
                assertions=[session["assertions"]],
            ),
        )
        custom_results[session["scenario_name"]] = results
        for r in results:
            score = r.get("value", "N/A")
            label = r.get("label", "")
            explanation = r.get("explanation", "")
            print(f"    Score: {score} ({label})")
            if explanation:
                print(f"    Reason: {explanation[:120]}")
    except Exception as e:
        print(f"    ERROR: {e}")
        custom_results[session["scenario_name"]] = []

    print()

print("Custom evaluation complete.")

## 14b. Trajectory-Efficiency Grader (Deterministic)

The evaluators so far measure **whether** the answer is right. None of them measure **how efficiently** the agent got there. That is a real blind spot: at a workshop, "What are the items with low stock?" answered correctly but only after ~7 `odata_caller` calls (rediscovering the schema and guessing `$filter` syntax on every turn) — and `Correctness`/`GoalSuccessRate` still scored it 1.0.

This is exactly the kind of check the [Anthropic agent-eval guidance](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) calls a **code-based grader**: fast, cheap, objective, reproducible. We don't need an LLM to count tool calls.

**How it works:** the deployed agent already emits one OpenTelemetry span per tool invocation to CloudWatch (the same spans the built-in evaluators read). We query the log group for each session, count the `odata_caller` spans, and grade the count against the scenario's `max_tool_calls` budget. We also flag **redundancy signals** — repeated `$metadata` discovery calls — which are the fingerprint of the low-stock failure.

A scenario can **pass Correctness but fail efficiency** — which is the signal we want. Reading the flagged transcripts then points at concrete fixes (put the field names and a working `$filter` example in the system prompt, or give `odata_caller` a schema hint) that a "Correctness: 1.0" score would never surface.

In [ ]:
# Deterministic trajectory-efficiency grader: counts odata_caller spans per session
# from the OTEL logs already in CloudWatch, and grades against each scenario's budget.

logs_client = boto3.client("logs", region_name=REGION)

# Name of the tool we want to count. TODO: change if your agent uses a different tool.
TOOL_NAME = "odata_caller"


def _iter_tool_spans(raw_message: str):
    """Yield (span_key, text_blob) for tool-execution spans in one raw log event.

    OTEL span schemas vary, so we stay defensive: parse JSON when we can and fall
    back to raw-text matching. We only count spans that represent an actual tool
    invocation, not model turns that merely mention the tool name in text.
    """
    if TOOL_NAME not in raw_message:
        return
    try:
        doc = json.loads(raw_message)
        span_key = doc.get("spanId") or doc.get("span_id") or doc.get("traceId")
        name = str(doc.get("name", ""))
        attrs = json.dumps(doc.get("attributes", doc))
        is_tool_span = (
            "tool" in name.lower()
            or "gen_ai.tool.name" in attrs
            or '"tool.name"' in attrs
        )
        if is_tool_span:
            yield span_key, attrs
    except json.JSONDecodeError:
        # Not JSON — treat the raw line as a tool-span signal.
        yield None, raw_message


def grade_tool_efficiency(session_id: str, max_tool_calls: int):
    """Count tool spans for a session and grade against the budget.

    Returns a dict shaped like the evaluator results (evaluatorId/value/label/
    explanation) so it slots straight into the summary table.
    """
    events = []
    try:
        paginator = logs_client.get_paginator("filter_log_events")
        for page in paginator.paginate(
            logGroupName=CW_LOG_GROUP,
            filterPattern=f'"{session_id}"',
        ):
            events.extend(page.get("events", []))
    except Exception as e:
        return {
            "evaluatorId": "Custom.ToolCallEfficiency",
            "value": None,
            "label": "no_spans",
            "explanation": f"Could not read spans from {CW_LOG_GROUP}: {e}",
        }

    seen = set()
    tool_calls = 0
    metadata_calls = 0
    for ev in events:
        for span_key, blob in _iter_tool_spans(ev.get("message", "")):
            # Dedupe repeated log lines for the same span.
            key = span_key or blob[:200]
            if key in seen:
                continue
            seen.add(key)
            tool_calls += 1
            if "$metadata" in blob:
                metadata_calls += 1

    # Redundancy signals — the fingerprint of the workshop failure.
    redundancy = []
    if metadata_calls > 1:
        redundancy.append(f"{metadata_calls} $metadata discovery calls")

    within_budget = tool_calls <= max_tool_calls
    score = 1.0 if (within_budget and not redundancy) else 0.0

    if tool_calls == 0:
        label, explanation = "no_tool_spans", (
            "No tool spans found (spans may still be ingesting, or the session used no tools)."
        )
    elif score == 1.0:
        label = "efficient"
        explanation = f"{tool_calls} tool call(s), within budget of {max_tool_calls}."
    else:
        label = "inefficient"
        reasons = [f"{tool_calls} tool call(s) vs budget of {max_tool_calls}"] + redundancy
        explanation = "Inefficient trajectory: " + "; ".join(reasons) + "."

    return {
        "evaluatorId": "Custom.ToolCallEfficiency",
        "value": score,
        "label": label,
        "explanation": explanation,
        "n_tool_calls": tool_calls,
        "max_tool_calls": max_tool_calls,
    }


efficiency_results = {}

print("Running trajectory-efficiency grader on each session...\n")

for session in eval_sessions:
    result = grade_tool_efficiency(session["session_id"], session["max_tool_calls"])
    efficiency_results[session["scenario_name"]] = [result]
    print(f"  {session['scenario_name']}: {result['value']} ({result['label']})")
    print(f"    {result['explanation']}\n")

print("Trajectory-efficiency grading complete.")

## 15. Display Evaluation Results

Aggregate all scores into a summary table showing per-scenario, per-evaluator results.

In [ ]:
# Combine all results into a unified view (built-in + custom LLM judge + efficiency grader)
all_results = {}
for scenario_name in builtin_results:
    all_results[scenario_name] = (
        builtin_results.get(scenario_name, [])
        + custom_results.get(scenario_name, [])
        + efficiency_results.get(scenario_name, [])
    )

# Build score matrix
EFFICIENCY_EVALUATOR_ID = "Custom.ToolCallEfficiency"
all_evaluator_ids = BUILTIN_EVALUATOR_IDS + [CUSTOM_EVALUATOR_ID, EFFICIENCY_EVALUATOR_ID]


def _short_name(eid):
    if eid == CUSTOM_EVALUATOR_ID:
        return "OpsQuality"
    if eid == EFFICIENCY_EVALUATOR_ID:
        return "ToolEfficiency"
    return eid.split(".")[-1][:14]


print("=" * 106)
print(" AGENTCORE EVALUATION RESULTS")
print("=" * 106)

# Header
header = f"{'Scenario':<25}"
for eid in all_evaluator_ids:
    header += f" {_short_name(eid):>14}"
print(header)
print("-" * 106)

# Per-scenario scores
scenario_scores = defaultdict(dict)
for scenario_name, results in all_results.items():
    row = f"{scenario_name:<25}"
    for eid in all_evaluator_ids:
        score = next(
            (r.get("value", "-") for r in results if r.get("evaluatorId") == eid),
            "-",
        )
        if isinstance(score, (int, float)):
            row += f" {score:>14.2f}"
            scenario_scores[scenario_name][eid] = score
        else:
            row += f" {str(score):>14}"
    print(row)

# Averages
print("-" * 106)
avg_row = f"{'AVERAGE':<25}"
for eid in all_evaluator_ids:
    scores = [
        scenario_scores[s][eid]
        for s in scenario_scores
        if eid in scenario_scores[s]
    ]
    if scores:
        avg_row += f" {sum(scores)/len(scores):>14.2f}"
    else:
        avg_row += f" {'-':>14}"
print(avg_row)
print("=" * 106)

# Highlight the key insight: correct-but-inefficient trajectories.
# This is the low-stock failure mode — a right answer reached the wrong (wasteful) way.
print("\nCorrect-but-inefficient check (passes Correctness, fails ToolEfficiency):")
flagged = False
for scenario_name, scores in scenario_scores.items():
    if scores.get("Builtin.Correctness") == 1.0 and scores.get(EFFICIENCY_EVALUATOR_ID) == 0.0:
        flagged = True
        eff = next(
            (r for r in all_results[scenario_name]
             if r.get("evaluatorId") == EFFICIENCY_EVALUATOR_ID),
            {},
        )
        print(f"  [!] {scenario_name}: correct answer, but {eff.get('explanation', '')}")
if not flagged:
    print("  None flagged — every correct answer was also reached efficiently.")

# Show detailed results for one scenario
print(f"\nDetailed results for '{eval_sessions[0]['scenario_name']}':")
for r in all_results.get(eval_sessions[0]["scenario_name"], []):
    print(f"  {r.get('evaluatorId', 'unknown')}:")
    print(f"    Score: {r.get('value', 'N/A')} | Label: {r.get('label', 'N/A')}")
    if r.get("explanation"):
        print(f"    Explanation: {r['explanation'][:200]}")

## 16. Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources

# # Delete memory resource
# memory_client.delete_memory_and_wait(memory_id=memory_id)
# print(f"Deleted memory: {memory_id}")

# # Delete custom evaluator
# agentcore_control.delete_evaluator(evaluatorId=CUSTOM_EVALUATOR_ID)
# print(f"Deleted evaluator: {CUSTOM_EVALUATOR_ID}")

## Summary

In this lab, you built a production-ready warehouse operations agent that:

**Part A — Memory:**
- Uses **short-term memory** to maintain session continuity (no repeated context)
- Uses **long-term semantic memory** to remember operational facts across sessions
- Uses **long-term user preference memory** to personalize responses (formatting, thresholds)
- Integrates memory via Strands hooks using the `MemorySessionManager` pattern

**Part B — Evaluation:**
- Created a **custom LLM-as-a-judge evaluator** (WarehouseOperationalQuality) for domain-specific scoring
- Ran **built-in evaluators** (Correctness, Helpfulness, GoalSuccessRate, ToolSelectionAccuracy) via `EvaluationClient`
- Added a **deterministic trajectory-efficiency grader** that counts tool-call spans against a per-scenario budget — catching *correct-but-inefficient* answers (the low-stock query that used many redundant OData calls) that the built-in evaluators score 1.0
- Scored the deployed agent **on-demand** by invoking it with test scenarios and evaluating the resulting OTEL traces

**Key Takeaway:** AgentCore Evaluations reads OTEL traces from your deployed agent — no separate evaluation infrastructure needed. Combine **model-based** evaluators (correctness, helpfulness, domain quality) with cheap **code-based** graders (tool-call efficiency) so you measure not just *whether* the agent is right but *how efficiently* it gets there. Register evaluators once, then score any session on-demand or set up continuous online evaluation for production monitoring.

**Next steps (see the evaluation harness spec):** negative / out-of-scope scenarios, multi-trial runs with pass@k / pass^k, deterministic hallucination checks, and a pass/fail regression gate.